In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.decomposition import PCA

Load datasets created in R

In [2]:
norm_counts = pd.read_csv("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis/STAR_normalized_counts_new_filtering.csv", index_col=0)
norm_counts

,ERR12356072,ERR12383247,ERR12383248,ERR12383249,ERR12383250,ERR12383251,ERR12383252,ERR12383253,ERR12383254,ERR12383255,...,ERR12383308,ERR12383309,ERR12383310,ERR12383311,ERR12383312,ERR12383313,ERR12383314,ERR12383315,ERR12383316,ERR12383317
g1.t1,3.525821,4.498286,4.242980,4.067652,4.036978,4.201009,4.045398,3.525821,3.525821,4.304652,...,3.525821,4.212450,3.525821,4.113600,4.006012,4.232202,3.525821,4.495104,3.525821,3.525821
g10.t1,8.815704,8.927850,8.792050,9.662253,9.568433,9.183882,8.941532,8.557841,8.450497,8.922775,...,9.721615,9.399876,9.875382,9.056912,9.049214,9.144924,8.898725,9.268105,9.211291,8.875672
g1000.t1,11.125022,10.652687,10.964209,11.001276,11.362215,10.767366,10.449851,11.298761,10.436598,11.314721,...,10.724253,10.957839,10.630373,10.476187,10.177347,10.224182,10.315414,11.486511,11.873138,11.624127
g10001.t1,4.554830,4.352015,4.695109,5.424978,5.208745,4.842391,4.549020,4.358276,3.525821,5.032990,...,5.356908,4.596979,4.782318,4.530514,4.510798,4.313665,4.164134,6.044540,4.957293,4.959223
g10006.t1,4.640409,4.169224,4.242980,4.067652,4.036978,4.349105,4.300080,4.208597,3.525821,4.736437,...,4.433094,4.739287,5.042935,4.445948,5.367611,5.149564,5.625806,3.525821,4.130060,3.525821
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
g999.t1,4.891385,5.004564,4.959790,5.002116,4.532919,6.044083,5.116480,5.099237,4.876446,4.736437,...,5.263464,5.590934,5.405622,5.534265,5.461566,5.133458,5.649197,5.149383,5.111665,5.146642
g9991.t1,4.460042,4.562758,4.148469,4.710924,4.532919,4.349105,4.210435,3.525821,4.266261,4.079839,...,5.356908,4.646829,4.427881,4.113600,4.791240,4.702648,4.048401,4.873035,4.130060,3.938521
g9992.t1,4.292938,4.052591,4.325628,5.642992,4.923825,4.675333,4.379593,4.208597,4.427890,4.474150,...,3.985029,3.525821,3.525821,3.525821,3.525821,3.881647,3.525821,4.322062,4.557911,4.436597
g9993.t1,3.972170,4.428494,4.035511,3.525821,4.839450,5.789771,4.485339,4.482806,4.562072,5.828093,...,3.985029,3.525821,3.525821,4.530514,3.918494,3.525821,3.896345,4.092374,4.957293,5.269659


In [3]:
#Metadata
metadata = pd.read_csv("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis/dominance_meta_corrected_outlier_corrected.csv")

#make sure order matches norm_counts
metadata = metadata.set_index('Run')
metadata = metadata.reindex(norm_counts.columns)
# Now Run is the index, and metadata is aligned with counts
print("Metadata index (samples):", metadata.index[:5].tolist())
print("Counts columns:", norm_counts.columns[:5].tolist())
print("Match?", all(metadata.index == norm_counts.columns))
metadata

Metadata index (samples): ['ERR12356072', 'ERR12383247', 'ERR12383248', 'ERR12383249', 'ERR12383250']
Counts columns: ['ERR12356072', 'ERR12383247', 'ERR12383248', 'ERR12383249', 'ERR12383250']
Match? True


,Bases,BioProject,BioSample,Experiment,sample_name,TF ID,Sample ID,Cross,Family,Sex,original_fastq_name_R1,original_fastq_name_R2,Notes
ERR12356072,21027960416,PRJEB70958,SAMEA114860228,ERX11733017,Sample 1 males genotype BA heterozygote,TF2581-10-e3,10e 3,13:20 (M) + 42:13 (F),"2,3,4,",F,TF-2581-10-e3_S67_L001_R1_001.fastq.gz,TF-2581-10-e3_S67_L001_R2_001.fastq.gz,NaN
ERR12383247,14794943760,PRJEB70958,SAMEA114860213,ERX11759665,Sample 1 females genotype AA homozygote,TF2581-11,11,13:20 (M) + 42:13 (F),"2,3,4,",F,TF-2581-11_S9_L001_R1_001.fastq.gz,TF-2581-11_S9_L001_R2_001.fastq.gz,NaN
ERR12383248,15610248328,PRJEB70958,SAMEA114860214,ERX11759666,Sample 2 females genotype AA homozygote,TF2581-12,12,13:20 (M) + 42:13 (F),"2,3,4,",F,TF-2581-12_S10_L001_R1_001.fastq.gz,TF-2581-12_S10_L001_R2_001.fastq.gz,NaN
ERR12383249,9412089720,PRJEB70958,SAMEA114860215,ERX11759667,Sample 3 females genotype AA homozygote,TF2581-13,13,42:13 (M) + 13:20 (F),"1,4,8",M,TF-2581-13_S11_L001_R1_001.fastq.gz,TF-2581-13_S11_L001_R2_001.fastq.gz,NaN
ERR12383250,9959631122,PRJEB70958,SAMEA114860216,ERX11759668,Sample 1 males genotype AA homozygote,TF2581-14-e2,14e 2,42:13 (M) + 13:20 (F),"1,4,8",M,TF-2581-14-e2_S68_L001_R1_001.fastq.gz,TF-2581-14-e2_S68_L001_R2_001.fastq.gz,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ERR12383313,27406244206,PRJEB70958,SAMEA114860279,ERX11759731,Sample 1 females genotype FF homozygote,TF2581-71,71,4:18 + 4:18,"2,6,9",F,TF-2581-71_S59_L001_R1_001.fastq.gz,TF-2581-71_S59_L001_R2_001.fastq.gz,"Had TF ID: TF2581-70, and Sample ID: 71"
ERR12383314,13458064958,PRJEB70958,SAMEA114860280,ERX11759732,Sample 2 females genotype FF homozygote,TF2581-72,72,4:18 + 4:18,"2,6,9",F,TF-2581-72_S60_L001_R1_001.fastq.gz,TF-2581-72_S60_L001_R2_001.fastq.gz,"Had TF ID: TF2581-71, and Sample ID: 72"
ERR12383315,11890141660,PRJEB70958,SAMEA114860281,ERX11759733,Sample 3 females genotype FF homozygote,TF2581-7,7,13:20 (M) + 42:13 (F),"2,3,4,",M,TF-2581-7_S7_L001_R1_001.fastq.gz,TF-2581-7_S7_L001_R2_001.fastq.gz,NaN
ERR12383316,13692674262,PRJEB70958,SAMEA114860282,ERX11759734,Sample 1 males genotype FF homozygote,TF2581-8,8,13:20 (M) + 42:13 (F),"2,3,4,",M,TF-2581-8_S8_L001_R1_001.fastq.gz,TF-2581-8_S8_L001_R2_001.fastq.gz,NaN


PCA Plot

In [4]:
# create transpose
vst_t = norm_counts.T

pca = PCA(n_components=2)
pca_scores = pca.fit_transform(vst_t)

pca_df = metadata.copy()
pca_df['PC1'] = pca_scores[:,0]
pca_df['PC2'] = pca_scores[:,1]

# Explained variance
pc1_var = pca.explained_variance_ratio_[0] * 100
pc2_var = pca.explained_variance_ratio_[1] * 100

# Define your own color mapping
color_map = {'M': 'blue', 'F': 'red'}  

fig = px.scatter(
    pca_df,
    x='PC1',
    y='PC2',
    color='Sex',  
    color_discrete_map=color_map,          
    hover_name=pca_df.index,
    title=f"STAR-featureCounts: Male vs. Female PCA of VST-normalized counts"
)

fig.update_layout(
    xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)",
    yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)",
)
fig.show()


Volcano plot

In [5]:
results_full_annot = pd.read_csv("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis/STAR_featureCounts_deseq2_sex_results_fully_annotated_new_filtering.csv")
results_full_annot

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,start,...,HOG,OG,Gene.Tree.Parent.Clade,PFAMs,GOs,EC,KEGG_ko,KEGG_Pathway,COG_category,eggNOG_OGs
0,1.153465,-1.241867,0.542189,-2.290471,2.199401e-02,2.983143e-02,g1.t1,g1,utg000001l,185292,...,N0.HOG0006038,OG0005418,n0,-,"GO:0005575,GO:0005622,GO:0005623,GO:0005634,GO...",-,-,-,-,"2DQGE@1|root,2S6BS@2759|Eukaryota,3A6U0@33154|..."
1,456.100314,0.539063,0.128924,4.181259,2.898995e-05,5.207459e-05,g10.t1,g10,utg000001l,554496,...,N0.HOG0002621,OG0002163,n0,"EF-hand_1,EF-hand_5,EF-hand_6,EF-hand_7","GO:0003674,GO:0005488,GO:0005509,GO:0043167,GO...",-,-,-,T,"KOG0032@1|root,KOG0032@2759|Eukaryota,38G5X@33..."
2,1836.622862,0.664997,0.074970,8.870166,7.303700e-19,2.400314e-18,g1000.t1,g1000,utg000005l,5092168,...,N0.HOG0004899,OG0004294,n0,Ribonuclease_T2,"GO:0003674,GO:0003824,GO:0004518,GO:0004540,GO...",3.1.27.1,ko:K01166,-,A,"KOG1642@1|root,KOG1642@2759|Eukaryota,38KSQ@33..."
3,9.277403,1.333572,0.214887,6.205932,5.437368e-10,1.266580e-09,g10001.t1,g10001,utg000074l,8915422,...,N0.HOG0024445,OG0023792,-,DDE_Tnp_4,-,-,-,-,B,"KOG4585@1|root,KOG4585@2759|Eukaryota"
4,5.817209,-0.848882,0.462565,-1.835161,6.648186e-02,8.428873e-02,g10006.t1,g10006,utg000074l,9020033,...,N0.HOG0017991,OG0017338,n0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17561,20.318618,0.332159,0.149532,2.221320,2.632929e-02,3.535664e-02,g999.t1,g999,utg000005l,4837205,...,N0.HOG0007599,OG0006964,n0,Autophagy_act_C,"GO:0003674,GO:0003824,GO:0005575,GO:0005622,GO...",-,ko:K17888,"ko04136,ko04138,ko04140,map04136,map04138,map0...",S,"KOG4741@1|root,KOG4741@2759|Eukaryota,39WIU@33..."
17562,5.285936,0.872817,0.339448,2.571282,1.013228e-02,1.435931e-02,g9991.t1,g9991,utg000074l,8820390,...,N0.HOG0012813,OG0012161,n0,DDE_Tnp_4,-,-,-,-,L,"KOG4585@1|root,KOG4585@2759|Eukaryota,3A4U2@33..."
17563,3.573162,1.466915,0.329144,4.456757,8.320878e-06,1.547861e-05,g9992.t1,g9992,utg000074l,8822623,...,N0.HOG0013435,OG0012783,n0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17564,9.375622,0.551459,0.500326,1.102199,2.703753e-01,3.092871e-01,g9993.t1,g9993,utg000074l,8847120,...,N0.HOG0000071,OG0000033,n0,"BESS,MADF_DNA_bdg",-,-,-,-,K,"2E8EB@1|root,2SEWX@2759|Eukaryota,3AC3W@33154|..."


In [6]:
# Replace the zeros in padj with 1e-308 avoid log10 issues
results_full_annot["padj_safe"] = results_full_annot["padj"].replace(0, 1e-308).fillna(1)

# Add the negative log 10 padj for plotting
results_full_annot["neglog10_padj"] = -np.log10(results_full_annot["padj_safe"])


# Add significance to differentially expressed transcripts 
results_full_annot["significant"] = (
    (results_full_annot["padj"] < 0.05) &
    (results_full_annot["log2FoldChange"].abs() > 1)
)

# Add a label to the significant transcripts
results_full_annot["label"] = results_full_annot["transcript_id"].where(results_full_annot["significant"], "")

results_full_annot

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,start,...,GOs,EC,KEGG_ko,KEGG_Pathway,COG_category,eggNOG_OGs,padj_safe,neglog10_padj,significant,label
0,1.153465,-1.241867,0.542189,-2.290471,2.199401e-02,2.983143e-02,g1.t1,g1,utg000001l,185292,...,"GO:0005575,GO:0005622,GO:0005623,GO:0005634,GO...",-,-,-,-,"2DQGE@1|root,2S6BS@2759|Eukaryota,3A6U0@33154|...",2.983143e-02,1.525326,True,g1.t1
1,456.100314,0.539063,0.128924,4.181259,2.898995e-05,5.207459e-05,g10.t1,g10,utg000001l,554496,...,"GO:0003674,GO:0005488,GO:0005509,GO:0043167,GO...",-,-,-,T,"KOG0032@1|root,KOG0032@2759|Eukaryota,38G5X@33...",5.207459e-05,4.283374,False,
2,1836.622862,0.664997,0.074970,8.870166,7.303700e-19,2.400314e-18,g1000.t1,g1000,utg000005l,5092168,...,"GO:0003674,GO:0003824,GO:0004518,GO:0004540,GO...",3.1.27.1,ko:K01166,-,A,"KOG1642@1|root,KOG1642@2759|Eukaryota,38KSQ@33...",2.400314e-18,17.619732,False,
3,9.277403,1.333572,0.214887,6.205932,5.437368e-10,1.266580e-09,g10001.t1,g10001,utg000074l,8915422,...,-,-,-,-,B,"KOG4585@1|root,KOG4585@2759|Eukaryota",1.266580e-09,8.897367,True,g10001.t1
4,5.817209,-0.848882,0.462565,-1.835161,6.648186e-02,8.428873e-02,g10006.t1,g10006,utg000074l,9020033,...,NaN,NaN,NaN,NaN,NaN,NaN,8.428873e-02,1.074231,False,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17561,20.318618,0.332159,0.149532,2.221320,2.632929e-02,3.535664e-02,g999.t1,g999,utg000005l,4837205,...,"GO:0003674,GO:0003824,GO:0005575,GO:0005622,GO...",-,ko:K17888,"ko04136,ko04138,ko04140,map04136,map04138,map0...",S,"KOG4741@1|root,KOG4741@2759|Eukaryota,39WIU@33...",3.535664e-02,1.451529,False,
17562,5.285936,0.872817,0.339448,2.571282,1.013228e-02,1.435931e-02,g9991.t1,g9991,utg000074l,8820390,...,-,-,-,-,L,"KOG4585@1|root,KOG4585@2759|Eukaryota,3A4U2@33...",1.435931e-02,1.842867,False,
17563,3.573162,1.466915,0.329144,4.456757,8.320878e-06,1.547861e-05,g9992.t1,g9992,utg000074l,8822623,...,NaN,NaN,NaN,NaN,NaN,NaN,1.547861e-05,4.810268,True,g9992.t1
17564,9.375622,0.551459,0.500326,1.102199,2.703753e-01,3.092871e-01,g9993.t1,g9993,utg000074l,8847120,...,-,-,-,-,K,"2E8EB@1|root,2SEWX@2759|Eukaryota,3AC3W@33154|...",3.092871e-01,0.509638,False,


In [7]:
fig = px.scatter(
    results_full_annot,
    x="log2FoldChange",
    y="neglog10_padj",
    hover_name="transcript_id",
    hover_data=["padj", "HOG", "PFAMs"],  # Use the columns that already exist
    color="significant",
    color_discrete_map={True: "red", False: "blue"},
    title="STAR-featureCounts: Volcano Plot (Male vs Female Transcript Expression)"
)

fig.update_traces(textposition='top center', textfont_size=8)

fig.update_layout(
    xaxis_title="log2 Fold Change (male vs female)",
    yaxis_title="-log10(padj (FDR))",
)

# Horizontal FDR=0.05 cutoff
padj_cut = -np.log10(0.05)

fig.add_hline(
    y=padj_cut,
    line_dash="dash",
    line_color="grey",
    annotation_text="padj = 0.05",
    annotation_position="bottom right"
)

# Vertical log2FC cutoffs
fig.add_vline(
    x=-1,
    line_dash="dash",
    line_color="grey",
    annotation_text="log2FC = -1",
    annotation_position="top left"
)
fig.add_vline(
    x=1,
    line_dash="dash",
    line_color="grey",
    annotation_text="log2FC = 1",
    annotation_position="top right"
)

fig.show()


#DE transcripts
sig_counts = results_full_annot["significant"].sum()
higher_in_m = ((results_full_annot["significant"]) & 
               (results_full_annot["log2FoldChange"] > 0)).sum()
higher_in_f = ((results_full_annot["significant"]) & 
               (results_full_annot["log2FoldChange"] < 0)).sum()

print(f"Total significant transcripts: {sig_counts}")
print(f"Higher in males: {higher_in_m}")
print(f"Higher in females: {higher_in_f}")

Total significant transcripts: 6977
Higher in males: 4804
Higher in females: 2173


Significant genes

In [8]:
sig_genes = results_full_annot[results_full_annot["significant"]]

sig_genes

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,start,...,GOs,EC,KEGG_ko,KEGG_Pathway,COG_category,eggNOG_OGs,padj_safe,neglog10_padj,significant,label
0,1.153465,-1.241867,0.542189,-2.290471,2.199401e-02,2.983143e-02,g1.t1,g1,utg000001l,185292,...,"GO:0005575,GO:0005622,GO:0005623,GO:0005634,GO...",-,-,-,-,"2DQGE@1|root,2S6BS@2759|Eukaryota,3A6U0@33154|...",2.983143e-02,1.525326,True,g1.t1
3,9.277403,1.333572,0.214887,6.205932,5.437368e-10,1.266580e-09,g10001.t1,g10001,utg000074l,8915422,...,-,-,-,-,B,"KOG4585@1|root,KOG4585@2759|Eukaryota",1.266580e-09,8.897367,True,g10001.t1
13,503.721255,-1.340437,0.071018,-18.874597,1.845263e-79,2.806397e-78,g1003.t1,g1003,utg000005l,5141771,...,"GO:0000976,GO:0000977,GO:0001012,GO:0001067,GO...",-,"ko:K09228,ko:K09229",-,J,"KOG1721@1|root,KOG1721@2759|Eukaryota,38BZ9@33...",2.806397e-78,77.551851,True,g1003.t1
30,450.009283,6.846728,0.776649,8.815724,1.189137e-18,3.873238e-18,g10058.t1,g10058,utg000074l,9893841,...,NaN,NaN,NaN,NaN,NaN,NaN,3.873238e-18,17.411926,True,g10058.t1
31,27.548224,4.055556,0.243055,16.685778,1.663243e-62,1.753693e-61,g10059.t1,g10059,utg000074l,9899121,...,-,-,-,-,-,"2CMJZ@1|root,2QQM3@2759|Eukaryota,39MCP@33154|...",1.753693e-61,60.756046,True,g10059.t1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17554,3.472115,4.763003,0.721907,6.597809,4.172796e-11,1.025309e-10,g9961.t1,g9961,utg000074l,8549729,...,NaN,NaN,NaN,NaN,NaN,NaN,1.025309e-10,9.989145,True,g9961.t1
17558,3.644762,1.056110,0.347704,3.037382,2.386425e-03,3.643311e-03,g9975.t1,g9975,utg000074l,8597833,...,NaN,NaN,NaN,NaN,NaN,NaN,3.643311e-03,2.438504,True,g9975.t1
17559,65.202446,1.386464,0.282589,4.906290,9.281514e-07,1.830871e-06,g998.t1,g998,utg000005l,4824834,...,-,-,-,-,S,"KOG1075@1|root,KOG1075@2759|Eukaryota,3A0JU@33...",1.830871e-06,5.737342,True,g998.t1
17563,3.573162,1.466915,0.329144,4.456757,8.320878e-06,1.547861e-05,g9992.t1,g9992,utg000074l,8822623,...,NaN,NaN,NaN,NaN,NaN,NaN,1.547861e-05,4.810268,True,g9992.t1
